# 01 — Multilevel model of inter- and intra-strain variability (CmdStan)

Fits Equations 1a–1d of the manuscript to the plate counts of the 50 *C. jejuni*
strains heated at 55 °C (three replicate experiments per strain), and saves the
posterior draws that the simulations in notebooks 02 and 03 use.

| Level | What it describes | Output used by the simulation |
|---|---|---|
| Plate | Poisson scatter of colony counts around the true concentration (Eq. 1a) | — |
| Replicate (curve) | Replicate experiments of one strain differ: **intra-strain variability** (Eq. 1c) | `COV_intra` |
| Strain | Strains differ: **inter-strain variability** (Eq. 1d) | `mu_inter`, `COV_inter` |

**Relation to the original PyStan 2.19 fit** (notebook `...All_strain_MVN-Copy1.ipynb`, 2025)

The original fit had the same three levels (one (δ, p) per replicate curve, Eq. 1c; strain
means drawn from the inter-strain distribution, Eq. 1d). It additionally had (i) a second,
weakly identified layer around each strain mean (`within0`), (ii) the constraint that each
inter-strain SD exceeds the corresponding intra-strain SD, and (iii) narrower half-Cauchy priors
(0.1 intra, 0.5 inter). It sampled with 5.6 % divergent transitions and E-BFMI ≈ 0.1.
The model here is Eq. 1a–1d as written in the manuscript, without (i)–(iii); fitted to the same
data it reproduces the original estimates (section 6) and samples without divergences.

In [ ]:
import sys, os, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import campyevo
from campyevo import n, QUICK, DATA, RESULTS, FIGURES, TABLES, SIMS, POSTERIOR
from campyevo.plotting import setup_style, savefig
setup_style()
pd.set_option("display.float_format", lambda v: f"{v:.4g}")
warnings.simplefilter("ignore", RuntimeWarning)
print("project root:", ROOT, "| QUICK mode:", QUICK)

## 1. Settings

* `DATA_FILE`: the plate-count table (columns `Strain_Name, rep_ID, t_heat, n_dilu, V, Nt_count`),
  found in `data/`. If it is missing, the synthetic example table is used **for testing only**.
* `CMDSTAN_PATH`: leave `None` if CmdStan was installed with conda or `install_cmdstan()`.
* `OLD_PYSTAN_PICKLE`: optional, the posterior of the original fit, for comparison.

In [ ]:
from campyevo.data import find_dataset
DATA_FILE = find_dataset(DATA)        # the plate-count table shipped in data/
EXCLUDED = ("RIMD 0366027", "RIMD 0366043", "IFTK 344")   # removed in the original notebook
CMDSTAN_PATH = None
OLD_PYSTAN_PICKLE = DATA / "Samples extracted modfied Weibull multi-level MVN stan.pkl"

CHAINS = 4
ITER_WARMUP = n(2000, 500)
ITER_SAMPLING = n(2500, 500)      # 4 x 2,500 = 10,000 posterior draws
SEED = 12345

import cmdstanpy
if CMDSTAN_PATH:
    cmdstanpy.set_cmdstan_path(str(CMDSTAN_PATH))
print("CmdStan:", cmdstanpy.cmdstan_path(), "| cmdstanpy", cmdstanpy.__version__)

## 2. Data

In [ ]:
from campyevo.data import load_dataset, prepare

USING_SYNTHETIC = DATA_FILE is None
if USING_SYNTHETIC:
    print("!! data file not found -> using the SYNTHETIC example. Results are NOT the study's results.")
    cd = prepare(pd.read_csv(DATA / "synthetic_example.csv"), excluded=())
else:
    cd = load_dataset(DATA_FILE, excluded=EXCLUDED)

print(f"strains: {cd.n_strains}   curves (strain x replicate): {cd.n_curves}   plate counts: {len(cd.table)}")
reps = cd.summary()
print("replicates per strain:", reps["replicates"].value_counts().to_dict())
print("heating times (min):", sorted(cd.table["t_heat"].unique()))
reps.head()

The counts above are what Reviewer #1 asked to see stated in the Methods
(number of strains, replicate experiments per strain, sampling times, plates per time point).

In [ ]:
reps.to_csv(TABLES / "data_structure_per_strain.csv")

## 3. Fit

In [ ]:
from campyevo import fitting
model = fitting.compile_model()
fit = fitting.fit(cd.stan_data(), model=model, chains=CHAINS, iter_warmup=ITER_WARMUP,
                  iter_sampling=ITER_SAMPLING, seed=SEED, inits=cd.inits(),
                  output_dir=POSTERIOR / "cmdstan_csv")

In [ ]:
diag = fitting.diagnostics(fit)
print({k: v for k, v in diag.items() if k != "key_summary"})
diag["key_summary"]

Check before going on: no divergent transitions, R-hat < 1.01 for every quantity,
bulk ESS of the key quantities in the thousands.

## 4. Posterior summary (table for the manuscript)

In [ ]:
from campyevo.posterior import Posterior
post = Posterior.from_cmdstan(fit, source=("SYNTHETIC " if USING_SYNTHETIC else "") + "CmdStan " + cmdstanpy.cmdstan_version().__str__())
tab = post.summary()
tab.to_csv(TABLES / "TableS1b_posterior_summary.csv")
tab

In [ ]:
print(f"delta_mean = exp(mean ln delta_mean) = {post.delta_mean():.3f} min")
for R in (2, 4, 6):
    print(f"  {R}-log heating time = {R} x delta_mean = {R * post.delta_mean():.3f} min")

In [ ]:
post.save(POSTERIOR / ("posterior_synthetic.npz" if USING_SYNTHETIC else "posterior_cmdstan.npz"))
# Table S1a: the 50 strains, their data and their estimated parameters
tabS1a = (fitting.strain_table(fit, cd.strains)
          .merge(cd.summary().reset_index().rename(columns={"Strain_Name": "strain"}), on="strain", how="left"))
tabS1a.to_csv(TABLES / "TableS1a_strains.csv", index=False)

## 5. Figure 1 — present-day variability, from the data

(a) Measured survival curves of the least and the most heat-resistant strain: points are plate
counts, thin lines the fit to each replicate experiment, thick lines the strain. δ is shown as
it is defined: the time to a 6-log reduction divided by 6 (equal to the D-value when p = 1).
(b) All 50 strains on the δ–p plane with the range of present-day strains (90 %), and the three
replicate experiments of three strains: replicates of one strain lie closer together than strains
do. This within-strain spread is what the simulation uses for the change between generations.

In [ ]:
from campyevo import plotting as P
P.setup_style_v2()
fig = P.fig1_present_day_variability(cd, post)
savefig(fig, "Fig1_present_day_variability"); plt.show()

## 6. Comparison with the original PyStan posterior (optional)

If the pickle of the original fit is in `data/`, its species-level quantities are
shown next to the new fit. With the same data file the two agree closely.

In [ ]:
if OLD_PYSTAN_PICKLE.exists():
    old = Posterior.from_pystan_pickle(OLD_PYSTAN_PICKLE)
    cmp = pd.concat({"new (CmdStan)": post.summary()[["median", "q2_5", "q97_5"]],
                     "original (PyStan 2.19)": old.summary()[["median", "q2_5", "q97_5"]]}, axis=1)
    cmp.to_csv(TABLES / "posterior_new_vs_original.csv")
    display(cmp)
else:
    print("original pickle not found:", OLD_PYSTAN_PICKLE)